#Feature Engineering
##SupplyPrescript project


In [24]:
import pandas as pd
import numpy as np

print("Libraries imported successfully!")

Libraries imported successfully!


In [25]:
df = pd.read_csv("../Dataset/cleaned_dataset.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Dataset loaded successfully!
Shape: (20651, 33)


C:\Users\tanvi\AppData\Local\Temp\ipykernel_13476\3195554964.py:1: DtypeWarning: Columns (0: Project Code, 1: PQ #, 2: PO / SO #, 3: ASN/DN #, 4: Country, 5: Managed By, 6: Fulfill Via, 7: Vendor INCO Term, 8: Shipment Mode, 9: PQ First Sent to Client Date, 10: PO Sent to Vendor Date, 11: Scheduled Delivery Date, 12: Delivered to Client Date, 13: Delivery Recorded Date, 14: Product Group, 15: Sub Classification, 16: Vendor, 17: Item Description, 18: Molecule/Test Type, 19: Brand, 20: Dosage, 21: Dosage Form, 22: Manufacturing Site, 23: First Line Designation, 24: Weight (Kilograms), 25: Freight Cost (USD)) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../Dataset/cleaned_dataset.csv")


In [26]:
# Convert to real numbers, turning any leftover text into NaN
df["Weight (Kilograms)"] = pd.to_numeric(df["Weight (Kilograms)"], errors="coerce")
df["Freight Cost (USD)"] = pd.to_numeric(df["Freight Cost (USD)"], errors="coerce")

print("Converted successfully!")

Converted successfully!


In [27]:
df[["Weight (Kilograms)", "Freight Cost (USD)"]].isnull().sum()

Weight (Kilograms)    3955
Freight Cost (USD)    4129
dtype: int64

In [28]:
date_columns = ["Scheduled Delivery Date", "Delivered to Client Date", "Delivery Recorded Date"]

for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce")

print("Dates confirmed as datetime type!")

Dates confirmed as datetime type!


In [29]:
# Delay Days: how many days late (positive) or early (negative/zero) the shipment was
df["Delay Days"] = (df["Delivered to Client Date"] - df["Scheduled Delivery Date"]).dt.days

# Is Delayed: simple 1 (late) or 0 (on-time/early) flag
df["Is Delayed"] = (df["Delay Days"] > 0).astype(int)

print("Target variables created!")

Target variables created!


In [30]:
df["Is Delayed"].value_counts()


Is Delayed
0    19465
1     1186
Name: count, dtype: int64

In [31]:
df["Scheduled Month"] = df["Scheduled Delivery Date"].dt.month
df["Scheduled Year"] = df["Scheduled Delivery Date"].dt.year
df["Scheduled Weekday"] = df["Scheduled Delivery Date"].dt.dayofweek  # 0 = Monday, 6 = Sunday

print("Date features created!")
df[["Scheduled Delivery Date", "Scheduled Month", "Scheduled Year", "Scheduled Weekday"]].head()

Date features created!


,Scheduled Delivery Date,Scheduled Month,Scheduled Year,Scheduled Weekday
0,NaT,NaN,NaN,NaN
1,2006-06-02,6.0,2006.0,4.0
2,2006-11-14,11.0,2006.0,1.0
3,2006-08-27,8.0,2006.0,6.0
4,2006-09-01,9.0,2006.0,4.0


In [32]:
# Fill text columns
df["Shipment Mode"] = df["Shipment Mode"].fillna("Unknown")
df["Dosage"] = df["Dosage"].fillna("Not Specified")

# Fill number columns with the median
for col in ["Weight (Kilograms)", "Freight Cost (USD)", "Line Item Insurance (USD)"]:
    df[col] = df[col].fillna(df[col].median())

print("Missing values filled!")

Missing values filled!


In [33]:
df.isnull().sum().sort_values(ascending=False).head(10)

Scheduled Weekday           10327
Delay Days                  10327
Scheduled Month             10327
Delivery Recorded Date      10327
Delivered to Client Date    10327
Scheduled Delivery Date     10327
Scheduled Year              10327
Country                         3
First Line Designation          3
ASN/DN #                        3
dtype: int64

In [34]:
df.to_csv("../Dataset/feature_engineered_dataset.csv", index=False)

print("Feature engineered dataset saved successfully!")
print("Final shape:", df.shape)

Feature engineered dataset saved successfully!
Final shape: (20651, 38)
